# Column-Level Data Dictionary

This notebook walks through every raw dataset (excluding the four wide
gene-expression matrices that have 1000+ gene columns - those are tackled
separately) and explains, in plain language, **what each column means
biologically** and **why it might matter** for selecting a cell line.

For every dataset we build a small `pandas` DataFrame with three columns:

- `column` - the name of the field in the raw file
- `dtype` - the pandas data type (inferred from a sample of rows)
- `description` - a domain-expert explanation of what the field represents

These per-dataset dictionaries are the foundation for the preprocessing
pipeline: they tell us which columns carry biological signal (and should be
considered as features or justification text) versus which are just
bookkeeping/IDs.


In [1]:
import pandas as pd
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.max_rows', 300)


## 1. HPA RNA Cell Line Expression (`1_4_hpa_rna_celline.tsv`)

The Human Protein Atlas (HPA) measured, by RNA sequencing, how much each gene
is "switched on" (expressed) in ~1000 human cancer cell lines. The table is
in **long format**: one row = one gene measured in one cell line. This is the
most direct readout of *which genes a cell line actually uses*, and is the
core signal for matching a cell line to a disease of interest.


In [2]:
df1 = pd.read_csv('../../data/raw/gene_expression/1_4_hpa_rna_celline.tsv', sep='\t', nrows=5000)
desc1 = {
    'Gene': 'Ensembl gene ID (ENSG...) - the stable, version-independent identifier for the gene that was measured.',
    'Gene name': 'HGNC gene symbol (human-readable short name, e.g. TSPAN6) corresponding to the Ensembl ID.',
    'Cell line': 'Name of the cancer cell line in which expression was measured (links to cell-line nomenclature tables).',
    'TPM': 'Transcripts Per Million - raw normalized expression as output by the RNA-seq quantification tool.',
    'pTPM': 'Protein-coding TPM - TPM rescaled so the total sums to one million over protein-coding genes only (expression relative to the protein-coding transcriptome).',
    'nTPM': 'Normalized TPM - HPA batch-corrected value used for fair comparison of a gene\'s expression across different cell lines/experiments.',
}
dict1 = pd.DataFrame({'column': df1.columns, 'dtype': df1.dtypes.astype(str).values})
dict1['description'] = dict1['column'].map(desc1)
dict1


,column,dtype,description
0,Gene,str,"Ensembl gene ID (ENSG...) - the stable, version-independent identifier for the gene that was measured."
1,Gene name,str,"HGNC gene symbol (human-readable short name, e.g. TSPAN6) corresponding to the Ensembl ID."
2,Cell line,str,Name of the cancer cell line in which expression was measured (links to cell-line nomenclature tables).
3,TPM,float64,Transcripts Per Million - raw normalized expression as output by the RNA-seq quantification tool.
4,pTPM,float64,Protein-coding TPM - TPM rescaled so the total sums to one million over protein-coding genes only (expression relati...
5,nTPM,float64,Normalized TPM - HPA batch-corrected value used for fair comparison of a gene's expression across different cell lin...


## 2. Gene Fusion Events (`5_OmicsFusionFilteredSupplementary.csv`)

A **gene fusion** happens when a chromosomal rearrangement joins two
previously separate genes into one. Some fusions create powerful cancer
drivers (e.g. BCR-ABL in leukemia), so detecting them in RNA-seq data tells
us about a cell line's mutational background and which oncogenic pathways
are switched on. Each row is one candidate fusion detected in one sample.


In [3]:
df5 = pd.read_csv('../../data/raw/gene_properties/5_OmicsFusionFilteredSupplementary.csv', nrows=5000)
desc5 = {
    'Unnamed: 0': 'Row index carried over from the source export - no biological meaning.',
    'SequencingID': 'RNA-seq profile ID (PR-xxxx); links this fusion call back to the sequencing profile in DepMap_OmicsProfiles.',
    'ModelID': 'Cell line model identifier (ACH-xxxx); links to DepMap sample info.',
    'IsDefaultEntryForModel': 'Yes/No flag marking the canonical (preferred) profile DepMap uses to represent this model.',
    'ModelConditionID': 'Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.',
    'IsDefaultEntryForMC': 'Yes/No flag marking the canonical profile for that model condition.',
    'CanonicalFusionName': 'Gene1--Gene2 name of the fusion - the two genes that have been joined together.',
    'gene1(ENS ID)': "5' partner gene: its symbol plus its Ensembl gene ID.",
    'gene2(ENS ID)': "3' partner gene: its symbol plus its Ensembl gene ID.",
    'TotalReadsInSample': 'Total number of RNA-seq reads sequenced for this sample - a measure of sequencing depth.',
    'TotalReadsSupportingFusion': 'Number of reads that span or support the fusion junction - evidence strength for the fusion.',
    'FFPM': 'Fusion Fragments Per Million reads - normalized abundance of the fusion, comparable across samples of different depth.',
    'confidence': "The fusion-calling algorithm's confidence tier for this call: high, medium, or low.",
    'split_reads1': 'Number of reads with one segment aligning to gene 1 and the other spanning the breakpoint.',
    'split_reads2': 'Number of reads with one segment aligning to gene 2 and the other spanning the breakpoint.',
    'discordant_mates': 'Paired-end reads whose two mates map to the two different fusion partner genes - additional support for the fusion.',
    'strand1(gene/fusion)': "DNA strand orientation of gene 1 and of the resulting fusion transcript ('+' or '-').",
    'strand2(gene/fusion)': "DNA strand orientation of gene 2 and of the resulting fusion transcript ('+' or '-').",
    'reading_frame': 'Whether the fusion preserves an open reading frame (in-frame) or not (out-of-frame) - in-frame fusions are more likely to produce a functional fusion protein.',
    'breakpoint1': 'Genomic coordinate (chr:position) where gene 1 is broken to form the fusion.',
    'breakpoint2': 'Genomic coordinate (chr:position) where gene 2 is broken to form the fusion.',
    'site1': 'Genomic context of breakpoint 1 (e.g. CDS/splice-site, intergenic, UTR).',
    'site2': 'Genomic context of breakpoint 2 (e.g. CDS/splice-site, intergenic, UTR).',
    'type': 'Structural mechanism that produced the fusion (e.g. deletion/read-through, inversion, translocation).',
    'coverage1': 'Sequencing read depth covering the breakpoint region in gene 1.',
    'coverage2': 'Sequencing read depth covering the breakpoint region in gene 2.',
    'tags': 'Caller-assigned annotation flags; mostly a placeholder value ("." for no tag).',
    'retained_protein_domains': 'Protein domains that remain intact in the predicted fusion protein product.',
    'direction1': "Relative position (upstream/downstream) of gene 1's retained segment within the fusion transcript.",
    'direction2': "Relative position (upstream/downstream) of gene 2's retained segment within the fusion transcript.",
}
dict5 = pd.DataFrame({'column': df5.columns, 'dtype': df5.dtypes.astype(str).values})
dict5['description'] = dict5['column'].map(desc5)
dict5


,column,dtype,description
0,Unnamed: 0,int64,Row index carried over from the source export - no biological meaning.
1,SequencingID,str,RNA-seq profile ID (PR-xxxx); links this fusion call back to the sequencing profile in DepMap_OmicsProfiles.
2,ModelID,str,Cell line model identifier (ACH-xxxx); links to DepMap sample info.
3,IsDefaultEntryForModel,str,Yes/No flag marking the canonical (preferred) profile DepMap uses to represent this model.
4,ModelConditionID,str,Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.
5,IsDefaultEntryForMC,str,Yes/No flag marking the canonical profile for that model condition.
6,CanonicalFusionName,str,Gene1--Gene2 name of the fusion - the two genes that have been joined together.
7,gene1(ENS ID),str,5' partner gene: its symbol plus its Ensembl gene ID.
8,gene2(ENS ID),str,3' partner gene: its symbol plus its Ensembl gene ID.
9,TotalReadsInSample,int64,Total number of RNA-seq reads sequenced for this sample - a measure of sequencing depth.


## 3. Somatic Mutations (`6_OmicsSomaticMutationsProfile.csv`)

This is the most detailed dataset: every row is a single DNA mutation found
in a cell line's genome, together with dozens of annotations describing
*where* it is, *what* it changes, and *how dangerous* it is predicted to be
(from multiple independent prediction tools). Mutations in driver genes
(oncogenes/tumor suppressors) are often the **"WHY"** behind choosing a
particular cell line as a disease model - they recreate the genetic lesion
seen in patients.


In [4]:
df6 = pd.read_csv('../../data/raw/gene_properties/6_OmicsSomaticMutationsProfile.csv', nrows=5000, low_memory=False)
desc6 = {
    'Chrom': 'Chromosome on which the variant is located (e.g. chr1).',
    'Pos': 'Genomic position (1-based) of the variant on the chromosome.',
    'Ref': 'Reference allele - the sequence found in the normal human genome.',
    'Alt': 'Alternate (mutant) allele observed in this cell line.',
    'AF': "Allele/variant frequency - the fraction of sequencing reads carrying the mutant allele.",
    'DP': 'Total sequencing read depth at the variant position.',
    'RefCount': 'Number of reads supporting the reference (normal) allele.',
    'AltCount': 'Number of reads supporting the alternate (mutant) allele.',
    'GT': 'Genotype call, e.g. 0/1 = heterozygous, 1/1 = homozygous mutant.',
    'PS': 'Phase set ID - groups variants known to sit on the same chromosome copy (haplotype).',
    'VariantType': 'Class of variant: SNV (single nucleotide), insertion, deletion, substitution, etc.',
    'VariantInfo': "Sequence Ontology term(s) describing the variant's functional effect (e.g. missense_variant).",
    'DNAChange': 'HGVS notation describing the change at the DNA/transcript level (e.g. c.79_80delinsAA).',
    'ProteinChange': 'HGVS notation describing the resulting amino-acid change in the protein (e.g. p.A27K).',
    'HugoSymbol': 'HUGO gene symbol of the gene the mutation falls in.',
    'Exon': 'Which exon is affected, formatted as affected/total (e.g. 1/14).',
    'Intron': 'Which intron is affected, formatted as affected/total.',
    'EnsemblGeneID': 'Ensembl gene identifier of the affected gene.',
    'EnsemblFeatureID': 'Ensembl transcript identifier of the affected transcript.',
    'HgncName': 'Full descriptive name of the gene from HGNC.',
    'HgncFamily': 'HGNC gene family classification (groups genes by shared function/structure).',
    'UniprotID': 'UniProt protein accession/isoform affected by the variant.',
    'DbsnpRsID': 'dbSNP reference SNP ID, if this variant is a previously catalogued polymorphism.',
    'GcContent': 'Local GC nucleotide content around the variant - a sequencing/technical quality indicator.',
    'LofGeneName': 'Name of the gene predicted to lose function because of this variant.',
    'LofGeneId': 'Ensembl ID of the loss-of-function (LoF) gene.',
    'LofNumberOfTranscriptsInGene': 'Total number of annotated transcripts for that gene.',
    'LofPercentOfTranscriptsAffected': 'Fraction of the gene\'s transcripts predicted to be disrupted by this variant.',
    'NMD': 'Nonsense-mediated decay (NMD) prediction details - whether the mutant transcript is likely to be degraded.',
    'MolecularConsequence': 'Sequence Ontology ID and term(s) describing the molecular consequence of the variant.',
    'VepImpact': 'Ensembl VEP severity tier: HIGH, MODERATE, LOW, or MODIFIER.',
    'VepBiotype': 'Biotype of the affected transcript (protein_coding, lncRNA, pseudogene, etc.).',
    'VepHgncID': 'HGNC identifier of the affected gene, as reported by VEP.',
    'VepExistingVariation': 'IDs of known variants (dbSNP/COSMIC) overlapping this position.',
    'VepManeSelect': 'MANE Select transcript - the single, clinically-recommended reference transcript for the gene.',
    'VepENSP': 'Ensembl protein ID (ENSP) of the affected protein.',
    'VepSwissprot': 'UniProt/Swiss-Prot accession of the affected protein.',
    'Sift': 'SIFT prediction of how damaging the amino-acid change is (score + tolerated/deleterious label).',
    'Polyphen': 'PolyPhen-2 prediction of the structural impact on the protein (benign/possibly/probably damaging + score).',
    'GnomadeAF': 'Allele frequency of this variant in the gnomAD exomes population database.',
    'GnomadgAF': 'Allele frequency of this variant in the gnomAD genomes population database.',
    'VepClinSig': 'ClinVar clinical-significance annotation for this variant (e.g. pathogenic, benign).',
    'VepSomatic': 'Flags whether any known overlapping variants are annotated as somatic (cancer-acquired) rather than germline.',
    'VepPliGeneValue': "gnomAD pLI score - probability that this gene is intolerant of loss-of-function mutations.",
    'VepLofTool': 'LoFtool score - a gene-level measure of how intolerant the gene is to loss-of-function variation.',
    'OncogeneHighImpact': 'True/False flag marking this as a high-impact mutation in a known oncogene.',
    'TumorSuppressorHighImpact': 'True/False flag marking this as a high-impact mutation in a known tumor-suppressor gene.',
    'TranscriptLikelyLof': 'List of transcript IDs predicted to be loss-of-function as a result of this variant.',
    'Brca1FuncScore': 'Functional assay score for BRCA1 variants from saturation genome editing experiments.',
    'CivicID': 'Identifier of this variant in the CIViC database of clinically interpreted cancer variants.',
    'CivicDescription': "Curated free-text description from CIViC explaining this variant's clinical relevance.",
    'CivicScore': 'CIViC evidence score - reflects how much/strong clinical evidence exists for this variant.',
    'LikelyLoF': 'Overall True/False call for whether this variant is likely loss-of-function.',
    'HessDriver': 'True/False flag marking this as a likely cancer driver mutation per the Hess et al. signature analysis.',
    'HessSignature': 'Mutational signature(s) (e.g. UV, POLE, CpG) attributed to this variant by Hess et al.',
    'RevelScore': 'REVEL ensemble pathogenicity score for missense variants (combines multiple predictors).',
    'PharmgkbId': 'PharmGKB identifier linking this variant to pharmacogenomic (drug-response) annotations.',
    'DidaID': 'Identifier in the DIDA database of digenic disease associations.',
    'DidaName': 'Disease name associated with this variant via the DIDA database.',
    'GwasDisease': 'Trait or disease this variant has been associated with in genome-wide association studies (GWAS).',
    'GwasPmID': 'PubMed ID of the GWAS publication reporting that association.',
    'GtexGene': 'Gene associated with this variant via GTEx expression quantitative trait loci (eQTL) data.',
    'ProveanPrediction': 'PROVEAN prediction of the variant\'s effect on protein function (Neutral/Damaging).',
    'AMClass': 'AlphaMissense pathogenicity classification (likely_benign / ambiguous / likely_pathogenic).',
    'AMPathogenicity': 'AlphaMissense continuous pathogenicity score, ranging from 0 (benign) to 1 (pathogenic).',
    'Rescue': 'True/False flag indicating the variant was retained ("rescued") by curation rules despite normally being filtered out.',
    'RescueReason': 'Reason code explaining why the variant was rescued/retained (e.g. Oncogene_high_impact).',
    'ProfileID': 'Sequencing profile ID this mutation was called from; links to DepMap_OmicsProfiles.',
    'Hotspot': 'True/False flag marking this position as a known recurrent mutational hotspot in cancer.',
    'EntrezGeneID': 'NCBI Entrez gene ID of the affected gene.',
}
dict6 = pd.DataFrame({'column': df6.columns, 'dtype': df6.dtypes.astype(str).values})
dict6['description'] = dict6['column'].map(desc6)
dict6


,column,dtype,description
0,Chrom,str,Chromosome on which the variant is located (e.g. chr1).
1,Pos,int64,Genomic position (1-based) of the variant on the chromosome.
2,Ref,str,Reference allele - the sequence found in the normal human genome.
3,Alt,str,Alternate (mutant) allele observed in this cell line.
4,AF,float64,Allele/variant frequency - the fraction of sequencing reads carrying the mutant allele.
5,DP,int64,Total sequencing read depth at the variant position.
6,RefCount,int64,Number of reads supporting the reference (normal) allele.
7,AltCount,int64,Number of reads supporting the alternate (mutant) allele.
8,GT,str,"Genotype call, e.g. 0/1 = heterozygous, 1/1 = homozygous mutant."
9,PS,float64,Phase set ID - groups variants known to sit on the same chromosome copy (haplotype).


## 4. Cellosaurus Cell Line Catalogue (`7_cellosaurus.csv`)

[Cellosaurus](https://www.cellosaurus.org/) is the master encyclopedia of
cell lines used in biomedical research. It is the "phone book" that lets us
resolve the many different names a cell line might be called across datasets
(GEO, DepMap, HPA, ...) down to one stable accession (`CVCL_xxxx`), and it
carries rich metadata about the donor, disease, and authenticity of each
line.


In [5]:
df7 = pd.read_csv('../../data/raw/nomenclature/7_cellosaurus.csv', nrows=5000, low_memory=False)
desc7 = {
    'Identifier (cell line name)': 'The primary, preferred name used to refer to this cell line.',
    'Accession (CVCL_xxxx)': 'The unique, stable Cellosaurus accession number for this cell line - the universal join key.',
    'Secondary accession number(s)': 'Older/merged Cellosaurus accessions that now redirect to this entry.',
    'Synonyms': 'Alternative names and spellings this cell line is also known by.',
    'Cross-references': "Identifiers for this cell line in other databases (e.g. Wikidata, ATCC, ECACC).",
    'References identifiers': 'Publications (PubMed) or patents that describe or establish this cell line.',
    'Web pages': 'External web links with further information about the cell line.',
    'Comments': 'Free-text curator notes: provenance, transformation status, antibody targets, group classification, etc.',
    'STR profile data': 'Short Tandem Repeat (STR) DNA fingerprint - used to verify the cell line\'s identity and detect cross-contamination/misidentification.',
    'Diseases': 'Disease(s) the cell line is associated with, coded against the NCI Thesaurus ontology.',
    'Species of origin': 'Species the cell line was derived from (e.g. Homo sapiens, Mus musculus), with NCBI taxonomy ID.',
    'Hierarchy': 'Parent cell line(s) this line was derived from, if it is a sub-clone or derivative.',
    'Originate from same individual': 'Other cell lines known to come from the same patient/donor.',
    'Sex of cell': 'Sex of the donor the cell line was derived from.',
    'Age of donor at sampling': "Donor's age (or developmental stage, e.g. Embryo) at the time the sample was taken.",
    'Category': 'High-level category of the cell line (e.g. Cancer cell line, Hybridoma, Transformed cell line).',
    'Date (entry history)': 'Creation date, last-update date, and version number of this Cellosaurus record.',
}
dict7 = pd.DataFrame({'column': df7.columns, 'dtype': df7.dtypes.astype(str).values})
dict7['description'] = dict7['column'].map(desc7)
dict7


,column,dtype,description
0,Identifier (cell line name),str,"The primary, preferred name used to refer to this cell line."
1,Accession (CVCL_xxxx),str,"The unique, stable Cellosaurus accession number for this cell line - the universal join key."
2,Secondary accession number(s),str,Older/merged Cellosaurus accessions that now redirect to this entry.
3,Synonyms,str,Alternative names and spellings this cell line is also known by.
4,Cross-references,str,"Identifiers for this cell line in other databases (e.g. Wikidata, ATCC, ECACC)."
5,References identifiers,str,Publications (PubMed) or patents that describe or establish this cell line.
6,Web pages,str,External web links with further information about the cell line.
7,Comments,str,"Free-text curator notes: provenance, transformation status, antibody targets, group classification, etc."
8,STR profile data,str,Short Tandem Repeat (STR) DNA fingerprint - used to verify the cell line's identity and detect cross-contamination/m...
9,Diseases,str,"Disease(s) the cell line is associated with, coded against the NCI Thesaurus ontology."


## 5. DepMap Omics Profiles (`8_DepMap_OmicsProfiles.csv`)

DepMap (the Cancer Dependency Map) sequences each cell line in multiple ways
- whole genome (WGS), whole exome (WES), and RNA. This small table is the
**index/lookup** that says, for every sequencing profile, which cell line it
came from and which technology was used. Other DepMap tables (mutations,
fusions, signatures) reference the `ProfileID` defined here.


In [6]:
df8 = pd.read_csv('../../data/raw/nomenclature/8_DepMap_OmicsProfiles.csv', nrows=5000)
desc8 = {
    'ProfileID': 'Unique sequencing profile identifier (PR-xxxx) - the join key used by mutation, fusion, and signature tables.',
    'ModelCondition': 'Identifier (MC-xxxx) for the specific experimental condition the cell line was grown/sequenced under.',
    'ModelID': 'Cell line model identifier (ACH-xxxx) - links to DepMap sample info.',
    'Datatype': 'Sequencing assay performed: wgs (whole genome), wes (whole exome), or rna (RNA-seq).',
    'WESKit': 'Exome capture kit used for whole-exome sequencing (e.g. AGILENT, ICE); empty for non-WES profiles.',
}
dict8 = pd.DataFrame({'column': df8.columns, 'dtype': df8.dtypes.astype(str).values})
dict8['description'] = dict8['column'].map(desc8)
dict8


,column,dtype,description
0,ProfileID,str,"Unique sequencing profile identifier (PR-xxxx) - the join key used by mutation, fusion, and signature tables."
1,ModelCondition,str,Identifier (MC-xxxx) for the specific experimental condition the cell line was grown/sequenced under.
2,ModelID,str,Cell line model identifier (ACH-xxxx) - links to DepMap sample info.
3,Datatype,str,"Sequencing assay performed: wgs (whole genome), wes (whole exome), or rna (RNA-seq)."
4,WESKit,str,"Exome capture kit used for whole-exome sequencing (e.g. AGILENT, ICE); empty for non-WES profiles."


## 6. DepMap Sample Info (`9_DepMap_sample_info.csv`)

This is the **central metadata table** for every cell line in DepMap: who it
is, what cancer it represents, where it came from, and how it relates to
other lines. `DepMap_ID` is the primary key that almost every other DepMap
table (mutations, fusions, signatures, omics profiles) links back to. This
table is the natural place to look for the **"WHY"** - the disease context
that justifies selecting a given cell line.


In [7]:
df9 = pd.read_csv('../../data/raw/nomenclature/9_DepMap_sample_info.csv', nrows=5000, low_memory=False)
desc9 = {
    'DepMap_ID': 'Unique DepMap model identifier (ACH-xxxx) - the primary key linking all DepMap datasets.',
    'cell_line_name': 'Common/published name of the cell line.',
    'stripped_cell_line_name': 'Cell line name with spaces, dashes and punctuation removed - used for fuzzy matching across datasets.',
    'CCLE_Name': 'CCLE naming convention combining the cell line name with its tissue of origin (e.g. SLR21_KIDNEY).',
    'alias': 'Alternative name(s)/synonyms for the cell line.',
    'COSMICID': 'Identifier for this cell line in the COSMIC cancer mutation database.',
    'sex': 'Sex of the donor the cell line was derived from.',
    'source': 'Repository or lab that supplied the cell line (e.g. ATCC, DSMZ, Academic lab).',
    'RRID': 'Research Resource Identifier - the Cellosaurus accession (CVCL_xxxx) for this cell line.',
    'WTSI_Master_Cell_ID': 'Internal cell line identifier used by the Wellcome Sanger Institute.',
    'sample_collection_site': 'Anatomical site the tumor sample was collected from.',
    'primary_or_metastasis': 'Whether the sample was taken from the primary tumor or from a metastatic site.',
    'primary_disease': 'High-level cancer type/diagnosis (e.g. Kidney Cancer, Leukemia).',
    'Subtype': 'More specific disease subtype/histology (e.g. Renal Cell Carcinoma).',
    'age': 'Age of the donor at the time of sample collection.',
    'Sanger_Model_ID': 'Identifier for this cell line in the Sanger Cell Model Passports database.',
    'depmap_public_comments': 'Curator notes/caveats about the cell line, e.g. known misidentification or relationships to other lines.',
    'lineage': 'Broad tissue-of-origin lineage classification used for stratified analyses (e.g. kidney, blood, lung).',
    'lineage_subtype': 'More specific cancer subtype within the lineage (e.g. renal_cell_carcinoma, NSCLC).',
    'lineage_sub_subtype': 'Further refinement of the subtype classification (e.g. NSCLC_adenocarcinoma).',
    'lineage_molecular_subtype': 'Molecular subtype classification based on transcriptional/genomic profile (e.g. basal_B).',
    'default_growth_pattern': 'How the cell line grows in culture: adherent, suspension, or mixed.',
    'model_manipulation': 'Genetic or experimental manipulation applied to create this model, if any (e.g. immortalized, drug resistance selection).',
    'model_manipulation_details': 'Specific details of the manipulation (e.g. "STAG2 KO", "Drug resistance: Dabrafenib and Trametinib").',
    'patient_id': 'Identifier linking multiple cell lines derived from the same patient.',
    'parent_depmap_id': 'DepMap_ID of the parent line this model was derived from, for engineered derivatives.',
    'Cellosaurus_NCIt_disease': 'Disease term for this cell line, mapped via Cellosaurus to the NCI Thesaurus ontology.',
    'Cellosaurus_NCIt_id': 'NCI Thesaurus concept ID corresponding to the disease term.',
    'Cellosaurus_issues': 'Known data-quality or identity issues for this cell line as flagged in Cellosaurus.',
}
dict9 = pd.DataFrame({'column': df9.columns, 'dtype': df9.dtypes.astype(str).values})
dict9['description'] = dict9['column'].map(desc9)
dict9


,column,dtype,description
0,DepMap_ID,str,Unique DepMap model identifier (ACH-xxxx) - the primary key linking all DepMap datasets.
1,cell_line_name,str,Common/published name of the cell line.
2,stripped_cell_line_name,str,"Cell line name with spaces, dashes and punctuation removed - used for fuzzy matching across datasets."
3,CCLE_Name,str,CCLE naming convention combining the cell line name with its tissue of origin (e.g. SLR21_KIDNEY).
4,alias,str,Alternative name(s)/synonyms for the cell line.
5,COSMICID,float64,Identifier for this cell line in the COSMIC cancer mutation database.
6,sex,str,Sex of the donor the cell line was derived from.
7,source,str,"Repository or lab that supplied the cell line (e.g. ATCC, DSMZ, Academic lab)."
8,RRID,str,Research Resource Identifier - the Cellosaurus accession (CVCL_xxxx) for this cell line.
9,WTSI_Master_Cell_ID,float64,Internal cell line identifier used by the Wellcome Sanger Institute.


## 7. GEO Sample Info (`10_GEOInfo.txt`)

[GEO](https://www.ncbi.nlm.nih.gov/geo/) (Gene Expression Omnibus) is a
public archive of gene-expression experiments, mostly from older microarray
technology. This table is the metadata index for the GEO samples used in
`3_GEOexpression.txt` (the wide expression matrix we are deferring): it
records which GEO sample (`GSM` ID) corresponds to which cell line, and maps
that cell line to a stable Cellosaurus accession.


In [8]:
df10 = pd.read_csv('../../data/raw/nomenclature/10_GEOInfo.txt', sep='\t', nrows=5000, low_memory=False)
desc10 = {
    'Geo_accession': 'GEO sample accession (GSM number) - the unique identifier for one microarray sample.',
    'CEL_file_names': 'Name of the raw microarray (.CEL) data file associated with this sample.',
    'title': 'Free-text title given to the sample by the submitting lab.',
    'status': 'Public release status of the record in GEO, including the date it was made public.',
    'submission_date': 'Date the sample record was submitted to GEO.',
    'last_update_date': 'Date the GEO record was last updated.',
    'type': 'Type of molecule profiled (e.g. RNA).',
    'channel_count': 'Number of detection channels on the array; 1 means a single-channel (one-colour) array.',
    'source_name_ch1': 'Description of the biological source material loaded into channel 1 of the array.',
    'organism_ch1': 'Organism the sample was derived from.',
    'characteristics_ch1': 'Free-text key:value pairs describing sample characteristics (e.g. "gender: male", "cell line: ...").',
    'platform_id': 'GEO platform accession identifying the specific microarray chip used (e.g. GPL570).',
    'contact_country': 'Country of the laboratory that submitted the data.',
    'contact_institute': 'Institution that submitted the data.',
    'GSE_ID': 'GEO Series accession - groups all samples that belong to the same study.',
    'GSE_filename': 'Filename of the series matrix file containing this study\'s full expression data.',
    'cell_line': 'Cell line name exactly as reported by the submitting lab (may be inconsistent/non-standard).',
    'disease': 'Disease/diagnosis associated with the cell line as reported by the submitting lab.',
    'origin': 'Tissue of origin as reported by the submitting lab.',
    'Cellosaurus_ID': 'Cellosaurus accession (CVCL_xxxx) that this cell line has been matched to.',
    'Cellline': 'Standardized cell line name resolved via the Cellosaurus match.',
    'Matching_Type': 'How the reported cell line name was matched to a Cellosaurus entry (e.g. by GSM record, exact name, or synonym).',
    'cell_line_Trimmed': 'Lowercase, punctuation-stripped version of the cell line name, used as a join key for matching across datasets.',
}
dict10 = pd.DataFrame({'column': df10.columns, 'dtype': df10.dtypes.astype(str).values})
dict10['description'] = dict10['column'].map(desc10)
dict10


,column,dtype,description
0,Geo_accession,str,GEO sample accession (GSM number) - the unique identifier for one microarray sample.
1,CEL_file_names,str,Name of the raw microarray (.CEL) data file associated with this sample.
2,title,str,Free-text title given to the sample by the submitting lab.
3,status,str,"Public release status of the record in GEO, including the date it was made public."
4,submission_date,str,Date the sample record was submitted to GEO.
5,last_update_date,str,Date the GEO record was last updated.
6,type,str,Type of molecule profiled (e.g. RNA).
7,channel_count,float64,Number of detection channels on the array; 1 means a single-channel (one-colour) array.
8,source_name_ch1,str,Description of the biological source material loaded into channel 1 of the array.
9,organism_ch1,str,Organism the sample was derived from.


## 8. HPA Cell Line Description (`11_hpa_rna_celline_description.tsv`)

A short, curated metadata table for each cell line in the HPA RNA expression
dataset (Section 1). It supplies the disease context and donor information
needed to interpret the gene expression values - i.e. it answers "what
disease does this cell line model, and where did it come from?"


In [9]:
df11 = pd.read_csv('../../data/raw/nomenclature/11_hpa_rna_celline_description.tsv', sep='\t', nrows=5000)
desc11 = {
    'Cell line': 'Name of the cell line, matching the "Cell line" column in the HPA RNA expression dataset.',
    'Disease': 'Cancer type the cell line represents (e.g. Bone cancer, Prostate cancer).',
    'Disease subtype': 'More specific histological subtype of the disease (e.g. Osteosarcoma, Adenocarcinoma).',
    'Cellosaurus ID': 'Cellosaurus accession (CVCL_xxxx) for cross-referencing with other datasets.',
    'Patient': "Donor demographic information where known (age and/or sex), e.g. 'Male, 72'.",
    'Primary/Metastasis': 'Whether the source tumor sample was from the primary site or a metastasis.',
    'Sample collection site': 'Anatomical site the tumor sample was taken from.',
}
dict11 = pd.DataFrame({'column': df11.columns, 'dtype': df11.dtypes.astype(str).values})
dict11['description'] = dict11['column'].map(desc11)
dict11


,column,dtype,description
0,Cell line,str,"Name of the cell line, matching the ""Cell line"" column in the HPA RNA expression dataset."
1,Disease,str,"Cancer type the cell line represents (e.g. Bone cancer, Prostate cancer)."
2,Disease subtype,str,"More specific histological subtype of the disease (e.g. Osteosarcoma, Adenocarcinoma)."
3,Cellosaurus ID,str,Cellosaurus accession (CVCL_xxxx) for cross-referencing with other datasets.
4,Patient,str,"Donor demographic information where known (age and/or sex), e.g. 'Male, 72'."
5,Primary/Metastasis,str,Whether the source tumor sample was from the primary site or a metastasis.
6,Sample collection site,str,Anatomical site the tumor sample was taken from.


## 9. Omics Global Signatures (`14_OmicsGlobalSignatures.csv`)

These are **genome-wide summary statistics** computed from each cell line's
whole-genome/exome sequencing - single numbers that describe how "chaotic" a
cancer genome is overall (instability, ploidy, mismatch-repair status). They
are useful as quick, high-level features for comparing cell lines without
needing to look at individual mutations.


In [10]:
df14 = pd.read_csv('../../data/raw/non_gene_expression/14_OmicsGlobalSignatures.csv', nrows=5000)
desc14 = {
    'Unnamed: 0': 'Row index carried over from the source export - no biological meaning.',
    'SequencingID': 'WGS/WES profile ID (PR-xxxx); links to DepMap_OmicsProfiles.',
    'ModelID': 'Cell line model identifier (ACH-xxxx); links to DepMap sample info.',
    'ModelConditionID': 'Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.',
    'IsDefaultEntryForModel': 'Yes/No flag marking the canonical profile DepMap uses to represent this model.',
    'IsDefaultEntryForMC': 'Yes/No flag marking the canonical profile for that model condition.',
    'MSIScore': 'Microsatellite instability score; high values indicate a defective DNA mismatch-repair system, a known driver of hypermutation.',
    'LoHFraction': 'Fraction of the genome showing Loss of Heterozygosity (where one parental copy of a chromosome region has been lost).',
    'WGD': 'Whole-Genome Doubling flag (0/1) - whether the cell line\'s genome appears to have doubled in its evolutionary history.',
    'CIN': 'Chromosomal Instability score - summarizes how much copy-number variation/heterogeneity exists across the genome.',
    'Ploidy': 'Estimated average number of chromosome copies per cell (normal human cells are diploid, ploidy ~2).',
    'Aneuploidy': 'Score/count of chromosome arms with abnormal (non-diploid) copy number.',
}
dict14 = pd.DataFrame({'column': df14.columns, 'dtype': df14.dtypes.astype(str).values})
dict14['description'] = dict14['column'].map(desc14)
dict14


,column,dtype,description
0,Unnamed: 0,int64,Row index carried over from the source export - no biological meaning.
1,SequencingID,str,WGS/WES profile ID (PR-xxxx); links to DepMap_OmicsProfiles.
2,ModelID,str,Cell line model identifier (ACH-xxxx); links to DepMap sample info.
3,ModelConditionID,str,Identifier (MC-xxxx) for the specific experimental condition the sample was processed under.
4,IsDefaultEntryForModel,str,Yes/No flag marking the canonical profile DepMap uses to represent this model.
5,IsDefaultEntryForMC,str,Yes/No flag marking the canonical profile for that model condition.
6,MSIScore,float64,"Microsatellite instability score; high values indicate a defective DNA mismatch-repair system, a known driver of hyp..."
7,LoHFraction,float64,Fraction of the genome showing Loss of Heterozygosity (where one parental copy of a chromosome region has been lost).
8,WGD,float64,Whole-Genome Doubling flag (0/1) - whether the cell line's genome appears to have doubled in its evolutionary history.
9,CIN,float64,Chromosomal Instability score - summarizes how much copy-number variation/heterogeneity exists across the genome.


## 10. CCLE Metabolomics (`12_CCLE_metabolomics_20190502.csv`)

Metabolomics measures the levels of small-molecule chemicals ("metabolites")
- sugars, amino acids, lipids, nucleotides - inside each cell line. Unlike
gene expression (which tells us what a cell *could* do), metabolomics tells
us what the cell's biochemistry is *actually doing right now*. This is
valuable for finding cell lines with a distinctive metabolic phenotype (e.g.
high lactate / "Warburg effect" lines, or lines with an oncometabolite like
2-hydroxyglutarate).

There are 225 individual metabolites. Most are named directly; the lipid
species (the last ~88 columns) follow a shorthand naming convention,
`C<total carbons>:<double bonds> <lipid class>`, e.g. `C54:3 TAG` = a
triacylglycerol with 54 total carbon atoms and 3 double bonds across its
three fatty-acid chains. We decode both the named metabolites and the lipid
shorthand below.


In [11]:
df12 = pd.read_csv('../../data/raw/non_gene_expression/12_CCLE_metabolomics_20190502.csv', nrows=5000)

# --- 1. ID columns -----------------------------------------------------
desc12 = {
    'CCLE_ID': 'CCLE-style sample name combining the cell line name and tissue of origin (e.g. DMS53_LUNG).',
    'DepMap_ID': 'Unique DepMap model identifier (ACH-xxxx); links to DepMap sample info.',
}

# --- 2. Named (non-lipid) metabolites -----------------------------------
desc12.update({
    '2-aminoadipate': 'Lysine catabolism intermediate.',
    '3-phosphoglycerate': 'Glycolytic intermediate, three carbons downstream of glucose breakdown.',
    'alpha-glycerophosphate': 'Glycerol-3-phosphate; links glycolysis to lipid (triglyceride/phospholipid) synthesis.',
    '4-pyridoxate': 'Catabolite of vitamin B6 (pyridoxine).',
    'aconitate': 'TCA cycle intermediate, between citrate and isocitrate.',
    'adenine': 'Purine base; substrate for nucleotide salvage pathways.',
    'adipate': 'Dicarboxylic acid produced by fatty-acid omega-oxidation.',
    'alpha-ketoglutarate': 'TCA cycle intermediate; central hub linking energy metabolism, amino acid synthesis, and epigenetic enzyme activity.',
    'AMP': 'Adenosine monophosphate; reflects cellular energy charge and purine nucleotide pools.',
    'citrate': 'TCA cycle intermediate; also exported to the cytosol as the starting material for fatty-acid synthesis.',
    'isocitrate': 'TCA cycle intermediate, isomer of citrate.',
    'CMP': 'Cytidine monophosphate; pyrimidine nucleotide pool.',
    'cystathionine': 'Transsulfuration pathway intermediate converting methionine to cysteine.',
    'cytidine': 'Pyrimidine nucleoside.',
    'dCMP': 'Deoxy-CMP; a DNA-specific pyrimidine deoxynucleotide.',
    'DHAP/glyceraldehyde 3P': 'Triose-phosphate glycolytic intermediates (dihydroxyacetone phosphate / glyceraldehyde-3-phosphate).',
    'erythrose-4-phosphate': 'Pentose phosphate pathway intermediate that feeds aromatic amino acid biosynthesis.',
    'F1P/F6P/G1P/G6P': 'Fructose/glucose phosphate isomers; central glycolysis and glycogen metabolism intermediates.',
    'hexoses (HILIC neg)': 'Pooled six-carbon sugars (e.g. glucose, fructose) measured in negative-mode HILIC chromatography.',
    'fumarate/maleate/alpha-ketoisovalerate': 'TCA cycle intermediate (fumarate) co-eluting with a branched-chain amino acid catabolism intermediate.',
    'glucuronate': 'Glucuronic acid; used in detoxification (glucuronidation) and as a building block for glycosaminoglycans.',
    'glutathione oxidized': 'Oxidized glutathione (GSSG); together with reduced glutathione indicates cellular antioxidant/redox status.',
    'glutathione reduced': 'Reduced glutathione (GSH); the cell\'s main antioxidant, protects against oxidative damage.',
    'GMP': 'Guanosine monophosphate; purine nucleotide pool.',
    'guanosine': 'Purine nucleoside; nucleotide metabolism/salvage.',
    'hippurate': 'Glycine conjugate of benzoate, often linked to gut-microbiome or detoxification activity.',
    'hypoxanthine': 'Purine degradation intermediate, precursor of xanthine and uric acid.',
    'inosine': 'Purine nucleoside formed during purine degradation/salvage.',
    'kynurenine': 'First major intermediate of tryptophan breakdown via the kynurenine pathway; immune-modulatory.',
    'lactate': 'End product of anaerobic glycolysis; classic marker of the "Warburg effect" seen in many cancers.',
    'lactose': 'Disaccharide of glucose and galactose; typically a dietary/culture-medium residual, not made by human cells.',
    'malate': 'TCA cycle intermediate; also part of the malate-aspartate shuttle moving reducing power between cytosol and mitochondria.',
    'NAD': 'Nicotinamide adenine dinucleotide (oxidized form); central redox cofactor for energy metabolism.',
    'NADP': 'Nicotinamide adenine dinucleotide phosphate; redox cofactor that powers biosynthesis and antioxidant defenses.',
    'oxalate': 'Oxalic acid; a terminal metabolic waste product, relevant to kidney-stone biology.',
    'pantothenate': 'Vitamin B5; the direct precursor of coenzyme A.',
    'PEP': 'Phosphoenolpyruvate; the final high-energy intermediate of glycolysis before pyruvate.',
    'ribose-5-P/ribulose5-P': 'Pentose phosphate pathway intermediates; precursors for nucleotide synthesis and a source of NADPH.',
    'sorbitol': 'Sugar alcohol produced from glucose via the polyol pathway.',
    'succinate/methylmalonate': 'TCA cycle intermediate (succinate) co-eluting with a marker of branched-chain amino acid / odd-chain fatty acid catabolism.',
    'sucrose': 'Disaccharide of glucose and fructose; not endogenously produced by mammalian cells - a dietary/media residual.',
    'thymine': 'Pyrimidine base found specifically in DNA (not RNA).',
    'UMP': 'Uridine monophosphate; pyrimidine nucleotide and precursor of activated sugar donors (UDP-sugars).',
    'UDP-galactose/UDP-glucose': 'Activated sugar donors used for glycosylation and glycogen synthesis.',
    'uracil': 'Pyrimidine base specific to RNA; also a pyrimidine-degradation product.',
    'urate': 'Uric acid, the final product of purine degradation in humans; also acts as an antioxidant.',
    'uridine': 'Pyrimidine nucleoside; precursor for UTP and UDP-sugars.',
    'xanthine': 'Purine degradation intermediate, precursor of uric acid.',
    'taurocholate': 'Taurine-conjugated bile acid; involved in lipid digestion, normally produced in liver.',
    'glycodeoxycholate/glycochenodeoxycholate': 'Glycine-conjugated bile acids; involved in lipid digestion, normally produced in liver/gut.',
    'taurodeoxycholate/taurochenodeoxycholate': 'Taurine-conjugated secondary bile acids; involved in lipid digestion, normally produced in liver/gut.',
    'phosphocreatine': 'High-energy phosphate reservoir used to rapidly regenerate ATP.',
    '3-methyladipate/pimelate': 'Dicarboxylic acids arising from fatty-acid oxidation and biotin biosynthesis.',
    '6-phosphogluconate': 'Pentose phosphate pathway intermediate.',
    'alpha-hydroxybutyrate': 'Marker of altered cellular redox state / glutathione metabolism, also linked to insulin resistance.',
    '2-hydroxyglutarate': 'Oncometabolite produced by mutant IDH1/IDH2 enzymes; disrupts epigenetic regulation and is a hallmark of certain cancers.',
    'inositol': 'Sugar alcohol that is the structural backbone of phosphoinositide signalling lipids.',
    'malondialdehyde': 'Byproduct of lipid peroxidation; marker of oxidative stress.',
    'glycine': 'Simplest amino acid; central to one-carbon metabolism and glutathione synthesis.',
    'alanine': 'Amino acid that links glycolysis (via pyruvate) to amino acid metabolism.',
    'serine': 'Amino acid that is a key entry point into one-carbon/folate metabolism and nucleotide synthesis.',
    'threonine': 'Essential amino acid, catabolized to glycine and acetyl-CoA.',
    'methionine': 'Essential amino acid and methyl-group donor (via S-adenosylmethionine) for the methionine/homocysteine cycle.',
    'aspartate': 'Amino acid that feeds into the urea cycle and into purine/pyrimidine (nucleotide) synthesis.',
    'glutamate': 'Central amino acid linking the TCA cycle (via alpha-ketoglutarate) to amino acid metabolism.',
    'asparagine': 'Amide of aspartate; important for sustaining proliferation when glutamine is limited.',
    'glutamine': '"Conditionally essential" amino acid and major fuel source for rapidly proliferating (e.g. cancer) cells.',
    'histidine': 'Essential amino acid; precursor of histamine and involved in one-carbon metabolism.',
    'arginine': 'Semi-essential amino acid; precursor of nitric oxide and polyamines, and a urea-cycle intermediate.',
    'lysine': 'Essential amino acid; substrate for protein methylation/acetylation and for carnitine biosynthesis.',
    'valine': 'Branched-chain amino acid (BCAA); catabolized for energy, particularly in muscle and tumor tissue.',
    'leucine': 'Branched-chain amino acid (BCAA); activates mTOR signalling and promotes protein synthesis.',
    'isoleucine': 'Branched-chain amino acid (BCAA); catabolized via propionyl-CoA into the TCA cycle.',
    'phenylalanine': 'Aromatic amino acid; precursor of tyrosine.',
    'tyrosine': 'Aromatic amino acid; precursor of catecholamines and thyroid hormones.',
    'tryptophan': 'Aromatic amino acid; precursor of serotonin, niacin, and the kynurenine pathway metabolites.',
    'proline': 'Amino acid that is highly abundant in collagen and is linked to cellular redox balance.',
    'cis/trans-hydroxyproline': 'Hydroxylated form of proline, abundant in collagen; a marker of extracellular matrix/collagen turnover.',
    'ornithine': 'Urea cycle intermediate; precursor of polyamines via ornithine decarboxylase.',
    'citrulline': 'Urea cycle intermediate; also a biomarker of nitric oxide synthase activity.',
    'taurine': 'Sulfur-containing amino acid derivative acting as an osmolyte, antioxidant, and bile-acid conjugate.',
    '5-HIAA': '5-hydroxyindoleacetic acid; the main breakdown product of serotonin.',
    'serotonin': 'Neurotransmitter/hormone derived from tryptophan.',
    'GABA': 'Gamma-aminobutyric acid; the major inhibitory neurotransmitter, derived from glutamate.',
    'acetylglycine': 'Acetylated form of glycine; minor amino-acid conjugate/excretion product.',
    'dimethylglycine': 'Intermediate of choline/betaine metabolism feeding into one-carbon metabolism.',
    'homocysteine': 'Sulfur-containing amino acid intermediate of the methionine cycle; elevated levels are linked to cardiovascular risk.',
    'SDMA/ADMA': 'Symmetric/asymmetric dimethylarginine; products of arginine methylation that inhibit nitric oxide synthase.',
    'NMMA': 'N-monomethylarginine; another methylated arginine derivative that inhibits nitric oxide synthase.',
    'allantoin': 'Purine degradation product formed downstream of uric acid; a marker of oxidative stress.',
    'anthranilic acid': 'Kynurenine-pathway intermediate downstream of tryptophan.',
    'kynurenic acid': 'Neuroprotective metabolite of the kynurenine pathway.',
    '5-adenosylhomocysteine': 'S-adenosylhomocysteine (SAH); the byproduct left after SAM donates a methyl group - a key regulator of methylation reactions.',
    'carnosine': 'Dipeptide of beta-alanine and histidine; acts as an antioxidant and pH buffer, mainly in muscle.',
    'N-carbamoyl-beta-alanine': 'Intermediate of pyrimidine (uracil/thymine) degradation.',
    'thiamine': 'Vitamin B1; cofactor required by pyruvate dehydrogenase and other TCA-cycle-related enzymes.',
    'niacinamide': 'Vitamin B3 (nicotinamide); the direct precursor of NAD+.',
    'betaine': 'Choline-derived osmolyte and methyl-group donor.',
    'choline': 'Precursor of phosphatidylcholine (membrane lipid) and acetylcholine (neurotransmitter).',
    'alpha-glycerophosphocholine': 'Intermediate of choline/phospholipid metabolism, reflecting membrane turnover.',
    'acetylcholine': 'Neurotransmitter synthesized from choline.',
    'creatine': 'Energy-buffering molecule, part of the phosphocreatine system for rapid ATP regeneration.',
    'creatinine': 'Breakdown product of creatine; commonly used as a marker of renal function.',
    'thyroxine': 'Thyroid hormone (T4); presence in cell-line media is usually a culture-additive artifact rather than cell-intrinsic.',
    'trimethylamine-N-oxide': 'TMAO; a gut-microbiome-derived metabolite of choline, linked to cardiovascular risk.',
    'hexoses (HILIC pos)': 'Pooled six-carbon sugars (e.g. glucose, fructose) measured in positive-mode HILIC chromatography.',
    'adenosine': 'Purine nucleoside with signalling roles, and a nucleotide-metabolism intermediate.',
    'thymidine': 'Deoxyribonucleoside of thymine; a direct precursor for DNA synthesis and a proliferation marker.',
    'xanthosine': 'Purine nucleoside intermediate in purine catabolism.',
    '2-deoxyadenosine': 'Deoxyribonucleoside; a DNA precursor / purine salvage pathway intermediate.',
    '2-deoxycytidine': 'Deoxyribonucleoside; a DNA pyrimidine precursor.',
    'cAMP': 'Cyclic AMP; a key second messenger in intracellular signal transduction.',
    'cotinine': 'Metabolite of nicotine; indicates exposure to nicotine/tobacco (donor history or culture conditions).',
    'pipecolic acid': 'Intermediate of lysine catabolism.',
    'pyroglutamic acid': 'Cyclic derivative of glutamate formed as a byproduct of the glutathione cycle.',
    '1-methylnicotinamide': 'Methylated derivative of nicotinamide; reflects NAD+/niacin metabolism and methyl-group usage.',
    'butyrobetaine': 'Direct precursor in carnitine biosynthesis.',
    'putrescine': 'Polyamine produced by ornithine decarboxylase; associated with cell proliferation.',
    'methionine sulfoxide': 'Oxidized form of methionine; a marker of oxidative damage to proteins.',
    'sarcosine': 'N-methylglycine, a one-carbon metabolism intermediate reported to be elevated in some cancers (e.g. prostate).',
    'beta-alanine': 'Non-proteinogenic amino acid; precursor of coenzyme A (via pantothenate) and of carnosine.',
    'anserine': 'Dipeptide of beta-alanine and methylhistidine; a muscle buffer molecule, mostly of dietary origin.',
})

# --- 3. Acylcarnitines: carnitine conjugates of fatty acids, reflecting --
#        mitochondrial fatty-acid beta-oxidation flux ----------------------
for name in ['carnitine', 'acetylcarnitine', 'propionylcarnitine', 'malonylcarnitine',
              'butyrylcarnitine/isobutyrylcarnitine',
              'valerylcarnitine/isovalerylcarnitine/2-methylbutyroylcarnitine',
              'hexanoylcarnitine', 'heptanoylcarnitine', 'lauroylcarnitine',
              'myristoylcarnitine', 'palmitoylcarnitine', 'stearoylcarnitine',
              'oleylcarnitine', 'arachidonyl_carnitine']:
    if name == 'carnitine':
        desc12[name] = 'Required to shuttle fatty acids into mitochondria for beta-oxidation; the unconjugated "free" form.'
    else:
        desc12[name] = 'Acylcarnitine (a carnitine conjugated to a fatty acid chain); its level reflects mitochondrial fatty-acid beta-oxidation flux for that chain length.'

# --- 4. Lipid-class shorthand columns ------------------------------------
# Naming convention: "C<total carbons>:<double bonds> <CLASS>"
lipid_class_desc = {
    'LPC': 'Lysophosphatidylcholine - a membrane phospholipid breakdown product, marker of phospholipase activity.',
    'LPE': 'Lysophosphatidylethanolamine - a membrane phospholipid breakdown product.',
    'PC': 'Phosphatidylcholine - the most abundant membrane phospholipid.',
    'SM': 'Sphingomyelin - a membrane sphingolipid enriched in lipid rafts.',
    'DAG': 'Diacylglycerol - a lipid second messenger and intermediate in glycerolipid synthesis.',
    'CE': 'Cholesteryl ester - the storage form of cholesterol.',
    'TAG': 'Triacylglycerol - the main storage form of fat (energy reserve) in lipid droplets.',
}

for col in df12.columns:
    if col in desc12:
        continue
    # parse "Cxx:y CLASS" or "Cxx:y CLASS-A"
    try:
        chain, lipid_class = col.split(' ', 1)
        carbons, double_bonds = chain[1:].split(':')
        base_class = lipid_class.split('-')[0]
        class_text = lipid_class_desc[base_class]
        desc12[col] = (f'{class_text} This species has {carbons} total fatty-acid carbon '
                        f'atoms and {double_bonds} double bond(s) across its chain(s).')
    except (ValueError, KeyError):
        desc12[col] = 'Lipid species identified by mass spectrometry (see CCLE metabolomics methods for naming convention).'

dict12 = pd.DataFrame({'column': df12.columns, 'dtype': df12.dtypes.astype(str).values})
dict12['description'] = dict12['column'].map(desc12)
dict12


,column,dtype,description
0,CCLE_ID,str,CCLE-style sample name combining the cell line name and tissue of origin (e.g. DMS53_LUNG).
1,DepMap_ID,str,Unique DepMap model identifier (ACH-xxxx); links to DepMap sample info.
2,2-aminoadipate,float64,Lysine catabolism intermediate.
3,3-phosphoglycerate,float64,"Glycolytic intermediate, three carbons downstream of glucose breakdown."
4,alpha-glycerophosphate,float64,Glycerol-3-phosphate; links glycolysis to lipid (triglyceride/phospholipid) synthesis.
5,4-pyridoxate,float64,Catabolite of vitamin B6 (pyridoxine).
6,aconitate,float64,"TCA cycle intermediate, between citrate and isocitrate."
7,adenine,float64,Purine base; substrate for nucleotide salvage pathways.
8,adipate,float64,Dicarboxylic acid produced by fatty-acid omega-oxidation.
9,alpha-ketoglutarate,float64,"TCA cycle intermediate; central hub linking energy metabolism, amino acid synthesis, and epigenetic enzyme activity."


## 11. CCLE miRNA Expression (`13_CCLE_miRNA_20181103.gct`)

microRNAs (miRNAs) are short RNA molecules that fine-tune gene expression by
binding to messenger RNAs and blocking/degrading them. This file is in GCT
format (a standard for expression matrices): the first two columns identify
each miRNA, and **every other column is one cell line**, with the table cell
giving that miRNA's expression level in that cell line.

Because the 954 cell-line columns are all the *same kind* of thing (a sample
identifier + its expression values), we describe them as a single group
rather than repeating the same description 954 times.


In [12]:
df13 = pd.read_csv('../../data/raw/non_gene_expression/13_CCLE_miRNA_20181103.gct', sep='\t', skiprows=2, nrows=5000)

id_desc13 = {
    'Name': 'Internal probe identifier for the miRNA (format nmiR#####).',
    'Description': 'miRBase miRNA name (e.g. hsa-let-7a) - the biologically meaningful identifier for the microRNA.',
}

rows = []
for col in df13.columns:
    if col in id_desc13:
        rows.append({'column': col, 'dtype': str(df13[col].dtype), 'description': id_desc13[col]})

n_samples = len(df13.columns) - len(id_desc13)
sample_dtype = str(df13[df13.columns[2]].dtype) if len(df13.columns) > 2 else 'float64'
rows.append({
    'column': f'<954 cell-line columns, e.g. {df13.columns[2]}, {df13.columns[3]}, ...>',
    'dtype': sample_dtype,
    'description': ('Each remaining column is one CCLE cell line, named CELLLINE_TISSUE. '
                     'Cell values are that miRNA\'s normalized expression level (reads-per-million-style) '
                     f'in that cell line. There are {n_samples} such sample columns.'),
})

dict13 = pd.DataFrame(rows)
dict13


,column,dtype,description
0,Name,str,Internal probe identifier for the miRNA (format nmiR#####).
1,Description,str,miRBase miRNA name (e.g. hsa-let-7a) - the biologically meaningful identifier for the microRNA.
2,"<954 cell-line columns, e.g. DMS53_LUNG, SW1116_LARGE_INTESTINE, ...>",float64,"Each remaining column is one CCLE cell line, named CELLLINE_TISSUE. Cell values are that miRNA's normalized expressi..."


## Summary

| # | Dataset | File | Columns | Notes |
|---|---------|------|---------|-------|
| 1 | HPA RNA Cell Line Expression | `1_4_hpa_rna_celline.tsv` | 6 | Long format: gene x cell line x expression |
| 2 | Gene Fusions | `5_OmicsFusionFilteredSupplementary.csv` | 29 | One row per candidate fusion |
| 3 | Somatic Mutations | `6_OmicsSomaticMutationsProfile.csv` | 70 | One row per mutation, heavily annotated |
| 4 | Cellosaurus Catalogue | `7_cellosaurus.csv` | 17 | Master cell-line ID/metadata reference |
| 5 | DepMap Omics Profiles | `8_DepMap_OmicsProfiles.csv` | 5 | Sequencing profile index |
| 6 | DepMap Sample Info | `9_DepMap_sample_info.csv` | 29 | Central cell-line metadata table |
| 7 | GEO Sample Info | `10_GEOInfo.txt` | 23 | Microarray sample metadata index |
| 8 | HPA Cell Line Description | `11_hpa_rna_celline_description.tsv` | 7 | Disease context for HPA expression data |
| 9 | Omics Global Signatures | `14_OmicsGlobalSignatures.csv` | 12 | Genome-wide instability/ploidy summary stats |
| 10 | CCLE Metabolomics | `12_CCLE_metabolomics_20190502.csv` | 227 | 225 metabolites incl. 88 lipid species |
| 11 | CCLE miRNA Expression | `13_CCLE_miRNA_20181103.gct` | 956 | 2 ID columns + 954 cell-line samples |

**Deferred (>1000 columns - gene matrices, handled separately):**
`2_DepMap_OmicsExpressionAllGenesTPMLogp1Profile.csv` (53,962 cols),
`3_GEOexpression.txt` (3,268 cols), `4_Harmonized_MS_CCLE_Gygi_subsetted.csv` (12,559 cols).


In [13]:
# Create a list of all the dictionary DataFrames
dict_list = [
    dict1, dict5, dict6, dict7, dict8, 
    dict9, dict10, dict11, dict14, dict12, dict13
]

# List of corresponding dataset names for tracking
dataset_names = [
    'HPA RNA Cell Line Expression', 
    'Gene Fusion Events', 
    'Somatic Mutations', 
    'Cellosaurus', 
    'DepMap Omics Profiles', 
    'DepMap Sample Info', 
    'GEO Sample Info', 
    'HPA Cell Line Description', 
    'Omics Global Signatures', 
    'CCLE Metabolomics', 
    'CCLE miRNA Expression'
]

# Add a 'dataset' column to each dictionary so we know where each row came from
for df, name in zip(dict_list, dataset_names):
    df['dataset'] = name

# Concatenate all dictionaries into a single DataFrame
master_data_dict = pd.concat(dict_list, ignore_index=True)

# Reorder the columns for better readability and display the combined dictionary
master_data_dict = master_data_dict[['dataset', 'column', 'dtype', 'description']]
master_data_dict

,dataset,column,dtype,description
0,HPA RNA Cell Line Expression,Gene,str,"Ensembl gene ID (ENSG...) - the stable, version-independent identifier for the gene that was measured."
1,HPA RNA Cell Line Expression,Gene name,str,"HGNC gene symbol (human-readable short name, e.g. TSPAN6) corresponding to the Ensembl ID."
2,HPA RNA Cell Line Expression,Cell line,str,Name of the cancer cell line in which expression was measured (links to cell-line nomenclature tables).
3,HPA RNA Cell Line Expression,TPM,float64,Transcripts Per Million - raw normalized expression as output by the RNA-seq quantification tool.
4,HPA RNA Cell Line Expression,pTPM,float64,Protein-coding TPM - TPM rescaled so the total sums to one million over protein-coding genes only (expression relati...
...,...,...,...,...
424,CCLE Metabolomics,C58:7 TAG,float64,Triacylglycerol - the main storage form of fat (energy reserve) in lipid droplets. This species has 58 total fatty-a...
425,CCLE Metabolomics,C58:6 TAG,float64,Triacylglycerol - the main storage form of fat (energy reserve) in lipid droplets. This species has 58 total fatty-a...
426,CCLE miRNA Expression,Name,str,Internal probe identifier for the miRNA (format nmiR#####).
427,CCLE miRNA Expression,Description,str,miRBase miRNA name (e.g. hsa-let-7a) - the biologically meaningful identifier for the microRNA.


In [14]:
master_data_dict.to_excel("DataDictionary.xlsx", index=False)